In [ ]:
import numpy as np
from numpy.linalg import norm

# -----------------------------
# Lorenz with feedback
# -----------------------------
def lorenz_xyz(T=400_000, dt=6e-4, ρ=166.53, σ=10.0, β=8/3, k=0.0, H=25.0):
    steps = int(T)
    x = np.zeros((steps, 3), dtype=np.float32)
    x[0] = np.array([1.2, -1.3, 18.0])

    for t in range(steps-1):
        X, Y, Z = x[t]
        ctrl = -k * np.tanh((Z - H) / H)  # stabilization only when Z is large
        dX = σ*(Y - X)
        dY = X*(ρ - Z) - Y
        dZ = X*Y - β*Z + ctrl
        x[t+1] = x[t] + dt*np.array([dX, dY, dZ])
    return x

# -----------------------------
# Lyapunov exponent proxy (fast, stable)
# -----------------------------
def lyap(Z, down=50):
    d = np.abs(np.diff(Z))
    d = d[::down]
    d = d[d>1e-12]
    if len(d)<200:
        return np.nan
    return np.mean(np.log(d/d.mean()))

def DKY_from_l1(l1):
    return 1 + max(l1, 0)

# -----------------------------
# Laminar return-time tail exponent (robust)
# -----------------------------
def tau_laminar(X, p_in=0.35, hyster=1.45, min_episodes=40):
    r = np.sqrt(np.sum(X*X, axis=1))
    q = np.quantile(r, p_in)
    hi = hyster*q

    inside = r < q
    outside = r > hi

    # detect *returns* to laminar pocket
    episodes = []
    count = 0
    in_laminar = False
    for t in range(len(r)):
        if inside[t]:
            if not in_laminar:
                in_laminar = True
                count = 1
            else:
                count += 1
        elif outside[t] and in_laminar:
            episodes.append(count)
            in_laminar = False

    if len(episodes) < min_episodes:
        return np.nan

    # tail fit: P(L > x) ~ x^{-(τ-1)} -> log-log slope
    e = np.array(sorted(episodes))
    x = np.log(e)
    y = np.log(1 - np.arange(len(e))/len(e))
    m, _ = np.polyfit(x[-200:], y[-200:], 1)
    return 1 - m  # standard EVT form

# -----------------------------
# δ metric
# -----------------------------
def delta_metric(DKY, tau):
    if np.isnan(tau) or np.isnan(DKY):
        return np.nan
    return (DKY - 1) * (tau - 2)

# -----------------------------
# Sweep
# -----------------------------
ρ = 166.53  # crisis intermittency parameter
ks = np.linspace(0.0, 0.040, 21)

print(f"\nρ = {ρ}   (crisis intermittency regime)\n")
for k in ks:
    X = lorenz_xyz(T=300_000, dt=6e-4, ρ=ρ, k=k)
    Z = X[:,2]
    l1 = lyap(Z)
    DKY = DKY_from_l1(l1)
    tau = tau_laminar(X)
    delta = delta_metric(DKY, tau)
    print(f"k={k:0.3f} | λ1={l1:0.5f}  DKY={DKY:0.5f}  τ={tau if not np.isnan(tau) else np.nan:0.3f}  δ={delta if not np.isnan(delta) else np.nan:0.3f}")


ρ = 166.53   (crisis intermittency regime)

k=0.000 | λ1=-0.38844  DKY=1.00000  τ=6.719  δ=0.000
k=0.002 | λ1=-0.38970  DKY=1.00000  τ=6.761  δ=0.000
k=0.004 | λ1=-0.38903  DKY=1.00000  τ=6.762  δ=0.000
k=0.006 | λ1=-0.38563  DKY=1.00000  τ=6.750  δ=0.000
k=0.008 | λ1=-0.38538  DKY=1.00000  τ=6.737  δ=0.000
k=0.010 | λ1=-0.38279  DKY=1.00000  τ=6.718  δ=0.000
k=0.012 | λ1=-0.38580  DKY=1.00000  τ=6.733  δ=0.000
k=0.014 | λ1=-0.38582  DKY=1.00000  τ=6.723  δ=0.000
k=0.016 | λ1=-0.38506  DKY=1.00000  τ=6.693  δ=0.000
k=0.018 | λ1=-0.38955  DKY=1.00000  τ=6.686  δ=0.000
k=0.020 | λ1=-0.38984  DKY=1.00000  τ=6.706  δ=0.000
k=0.022 | λ1=-0.38504  DKY=1.00000  τ=6.735  δ=0.000
k=0.024 | λ1=-0.39165  DKY=1.00000  τ=6.746  δ=0.000
k=0.026 | λ1=-0.38756  DKY=1.00000  τ=6.698  δ=0.000
k=0.028 | λ1=-0.38925  DKY=1.00000  τ=6.706  δ=0.000
k=0.030 | λ1=-0.38787  DKY=1.00000  τ=6.741  δ=0.000
k=0.032 | λ1=-0.38895  DKY=1.00000  τ=6.715  δ=0.000
k=0.034 | λ1=-0.39005  DKY=1.00000  τ=6.729  δ=0.000
k

In [ ]:
ρ = 200  # <- CHAOTIC REGIME (not laminar)

ks = np.linspace(0.0, 0.050, 21)

print(f"\nρ = {ρ}   (strong chaos regime)\n")
for k in ks:
    X = lorenz_xyz(T=350_000, dt=5.5e-4, ρ=ρ, k=k)
    Z = X[:,2]
    l1 = lyap(Z, down=45)
    DKY = DKY_from_l1(l1)
    tau = tau_laminar(X, p_in=0.33, hyster=1.4, min_episodes=50)
    delta = delta_metric(DKY, tau)
    print(f"k={k:0.3f} | λ1={l1:0.5f}  DKY={DKY:0.5f}  τ={tau if not np.isnan(tau) else np.nan:0.3f}  δ={delta if not np.isnan(delta) else np.nan:0.3f}")


ρ = 200   (strong chaos regime)

k=0.000 | λ1=-0.37311  DKY=1.00000  τ=19.134  δ=0.000
k=0.003 | λ1=-0.37856  DKY=1.00000  τ=19.102  δ=0.000
k=0.005 | λ1=-0.37214  DKY=1.00000  τ=18.898  δ=0.000
k=0.007 | λ1=-0.37983  DKY=1.00000  τ=18.900  δ=0.000
k=0.010 | λ1=-0.38155  DKY=1.00000  τ=19.139  δ=0.000
k=0.013 | λ1=-0.37618  DKY=1.00000  τ=19.505  δ=0.000
k=0.015 | λ1=-0.37553  DKY=1.00000  τ=19.021  δ=0.000
k=0.018 | λ1=-0.37200  DKY=1.00000  τ=18.813  δ=0.000
k=0.020 | λ1=-0.37700  DKY=1.00000  τ=19.327  δ=0.000
k=0.022 | λ1=-0.37902  DKY=1.00000  τ=19.087  δ=0.000
k=0.025 | λ1=-0.37673  DKY=1.00000  τ=18.541  δ=0.000
k=0.028 | λ1=-0.37460  DKY=1.00000  τ=18.890  δ=0.000
k=0.030 | λ1=-0.38873  DKY=1.00000  τ=19.917  δ=0.000
k=0.033 | λ1=-0.37779  DKY=1.00000  τ=18.871  δ=0.000
k=0.035 | λ1=-0.37717  DKY=1.00000  τ=19.351  δ=0.000
k=0.037 | λ1=-0.37691  DKY=1.00000  τ=19.382  δ=0.000
k=0.040 | λ1=-0.37460  DKY=1.00000  τ=19.064  δ=0.000
k=0.043 | λ1=-0.37722  DKY=1.00000  τ=19.562  δ=

In [ ]:
import numpy as np

# =========================
# 1) Lorenz + bounded feedback (RK4 integrator)
# =========================
def lorenz_rhs(state, sigma=10.0, rho=200.0, beta=8.0/3.0, k=0.0, H=25.0):
    x, y, z = state
    ctrl = -k * np.tanh((z - H) / H)                  # bounded stabilizer
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z + ctrl
    return np.array([dx, dy, dz], dtype=float)

def lorenz_jacobian(state, sigma=10.0, rho=200.0, beta=8.0/3.0, k=0.0, H=25.0):
    x, y, z = state
    # d/dz of ctrl = -k*(1/H)*sech^2((z-H)/H)
    sh = np.cosh((z - H)/H)
    sech2 = 1.0/(sh*sh)
    dctrl_dz = -k * (1.0/H) * sech2

    J = np.array([[-sigma,      sigma,           0.0      ],
                  [ rho - z,    -1.0,           -x       ],
                  [     y,        x,   -beta + dctrl_dz  ]], dtype=float)
    return J

def rk4_step(f, state, dt, *args, **kwargs):
    k1 = f(state, *args, **kwargs)
    k2 = f(state + 0.5*dt*k1, *args, **kwargs)
    k3 = f(state + 0.5*dt*k2, *args, **kwargs)
    k4 = f(state + dt*k3, *args, **kwargs)
    return state + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def integrate_lorenz(T_steps=400_000, dt=5e-4, rho=200.0, k=0.0, x0=(1.2, -1.3, 18.0)):
    x = np.empty((T_steps, 3), dtype=float)
    s = np.array(x0, dtype=float)
    for i in range(T_steps):
        s = rk4_step(lorenz_rhs, s, dt, 10.0, rho, 8.0/3.0, k, 25.0)
        x[i] = s
    # drop warmup
    cut = int(0.2*T_steps)
    return x[cut:], dt

# =========================
# 2) Benettin largest Lyapunov exponent (robust)
# =========================
def largest_lyapunov_benettin(x, dt, rho=200.0, k=0.0, renorm_every=10):
    """
    Evolve a tangent vector via Jacobian and accumulate growth.
    Assumes x is the state trajectory already integrated with RK4.
    """
    # small tangent vector
    v = np.array([1.0, 0.0, 0.0], dtype=float)
    v /= np.linalg.norm(v)

    logs = []
    s = x[0].copy()
    for i in range(1, len(x)):
        # one RK4 state step already done in x; we reuse x[i] as next state
        s = x[i]
        # tangent update via linearization (Euler for tangent is enough at small dt)
        J = lorenz_jacobian(s, 10.0, rho, 8.0/3.0, k, 25.0)
        v = v + dt * (J @ v)

        if (i % renorm_every) == 0:
            ln = np.linalg.norm(v)
            if ln <= 1e-20:
                # reset to avoid collapse
                v = np.array([1.0, 0.0, 0.0], dtype=float)
                continue
            logs.append(np.log(ln))
            v = v / ln

    if len(logs) < 10:
        return np.nan

    # Lyapunov exponent per unit time:
    # each renorm_every steps correspond to renorm_every*dt time
    return (np.sum(logs) / len(logs)) / (renorm_every * dt)

def DKY_from_l1(l1):
    return 1.0 + max(l1, 0.0)

# =========================
# 3) τ from laminar dwell times near wings (robust, adaptive)
# =========================
def tau_laminar_adaptive(X, rho=200.0, p_in=0.35, hyster=1.45, min_episodes=50, tail_keep=200):
    """
    Crisis/intermittency detector:
    - distance to nearest wing fixed point in 3D
    - adaptive inner radius (percentile) + hysteresis outer radius
    - laminar dwell lengths
    - tail exponent via log-CCDF slope on upper tail
    """
    beta = 8.0/3.0
    x0 = np.sqrt(beta*(rho-1.0))
    Cplus  = np.array([ x0,  x0,  rho-1.0])
    Cminus = np.array([-x0, -x0,  rho-1.0])

    dplus  = np.linalg.norm(X - Cplus , axis=1)
    dminus = np.linalg.norm(X - Cminus, axis=1)
    d = np.minimum(dplus, dminus)

    eps_in  = np.quantile(d, p_in)
    eps_out = eps_in * hyster

    # hysteretic laminar mask
    mask = np.zeros_like(d, dtype=bool)
    on = False
    for i in range(len(d)):
        if not on and d[i] < eps_in:
            on = True
        elif on and d[i] > eps_out:
            on = False
        mask[i] = on

    runs = []
    c = 0
    for v in mask:
        if v: c += 1
        else:
            if c>0:
                runs.append(c)
                c = 0
    if c>0:
        runs.append(c)

    if len(runs) < min_episodes:
        return np.nan

    runs = np.array(sorted(runs))
    # Tail via log-CCDF (rank) slope on top 'tail_keep' points
    tail = runs[-min(tail_keep, len(runs)):]
    x = np.log(tail + 1e-9)
    # empirical CCDF ranks
    ranks = np.arange(len(tail), 0, -1).astype(float)
    y = np.log(ranks / ranks[0] + 1e-12)

    if len(x) < 10:
        return np.nan

    m, b = np.polyfit(x, y, 1)
    # CCDF ~ x^{-(tau-1)}  => slope = -(tau-1)
    tau = 1 - m
    return float(tau)

def delta_metric(DKY, tau):
    if not (np.isfinite(DKY) and np.isfinite(tau)):
        return np.nan
    return (DKY - 1.0) * (tau - 2.0)

# =========================
# 4) δ-law sweep (chaotic Lorenz; proper λ1)
# =========================
rho = 200.0                         # strong-chaos Lorenz
ks  = np.linspace(0.0, 0.040, 21)   # stabilizer sweep

print(f"\nρ = {rho:.2f}   (strong chaos, Benettin λ₁, adaptive laminar τ)\n")

for k in ks:
    # integrate once
    X, dt = integrate_lorenz(T_steps=360_000, dt=5e-4, rho=rho, k=k, x0=(1.2, -1.3, 18.0))
    # robust λ1 via Benettin
    l1 = largest_lyapunov_benettin(X, dt, rho=rho, k=k, renorm_every=10)
    DKY = DKY_from_l1(l1)
    # τ from laminar dwell times near wings
    tau = tau_laminar_adaptive(X, rho=rho, p_in=0.35, hyster=1.45, min_episodes=50, tail_keep=200)
    dlt = delta_metric(DKY, tau)
    print(f"k={k:0.3f} | λ1={l1: .5f}  DKY={DKY: .5f}  τ={tau if np.isfinite(tau) else np.nan: .3f}  δ={dlt if np.isfinite(dlt) else np.nan: .3f}")


ρ = 200.00   (strong chaos, Benettin λ₁, adaptive laminar τ)

k=0.000 | λ1= 1.54013  DKY= 2.54013  τ= 2.841  δ= 1.295
k=0.002 | λ1= 1.57733  DKY= 2.57733  τ= 3.034  δ= 1.631
k=0.004 | λ1= 1.52987  DKY= 2.52987  τ= 3.074  δ= 1.644
k=0.006 | λ1= 1.53070  DKY= 2.53070  τ= 2.842  δ= 1.289
k=0.008 | λ1= 1.51788  DKY= 2.51788  τ= 2.551  δ= 0.837
k=0.010 | λ1= 1.54681  DKY= 2.54681  τ= 2.800  δ= 1.238
k=0.012 | λ1= 1.57593  DKY= 2.57593  τ= 2.904  δ= 1.425
k=0.014 | λ1= 1.51052  DKY= 2.51052  τ= 2.824  δ= 1.245
k=0.016 | λ1= 1.55655  DKY= 2.55655  τ= 2.948  δ= 1.476
k=0.018 | λ1= 1.55478  DKY= 2.55478  τ= 2.816  δ= 1.269
k=0.020 | λ1= 1.58683  DKY= 2.58683  τ= 2.575  δ= 0.913
k=0.022 | λ1= 1.56846  DKY= 2.56846  τ= 2.690  δ= 1.083
k=0.024 | λ1= 1.59129  DKY= 2.59129  τ= 2.654  δ= 1.041
k=0.026 | λ1= 1.47185  DKY= 2.47185  τ= 2.888  δ= 1.307
k=0.028 | λ1= 1.54437  DKY= 2.54437  τ= 2.813  δ= 1.256
k=0.030 | λ1= 1.53302  DKY= 2.53302  τ= 2.844  δ= 1.293
k=0.032 | λ1= 1.57178  DKY= 2.57178  τ= 2

In [ ]:
import numpy as np
import math
from math import sqrt
import sys
from collections import namedtuple

# =========================
# 1) Lorenz + bounded feedback (RK4 integrator)
# =========================
def lorenz_rhs(state, sigma=10.0, rho=200.0, beta=8.0/3.0, k=0.0, H=25.0):
    x, y, z = state
    ctrl = -k * np.tanh((z - H) / H)  # bounded stabilizer (URT-style saturating feedback)
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z + ctrl
    return np.array([dx, dy, dz], dtype=float)

def lorenz_jacobian(state, sigma=10.0, rho=200.0, beta=8.0/3.0, k=0.0, H=25.0):
    x, y, z = state
    # derivative of control wrt z: -k*(1/H)*sech^2((z-H)/H)
    s = (z - H)/H
    ch = np.cosh(s)
    sech2 = 1.0/(ch*ch)
    dctrl_dz = -k * (1.0/H) * sech2
    # Jacobian
    return np.array([
        [-sigma,    sigma,     0.0],
        [ rho - z,   -1.0,     -x ],
        [    y,        x,  -beta + dctrl_dz]
    ], dtype=float)

def rk4_step(f, state, dt, *args, **kwargs):
    k1 = f(state, *args, **kwargs)
    k2 = f(state + 0.5*dt*k1, *args, **kwargs)
    k3 = f(state + 0.5*dt*k2, *args, **kwargs)
    k4 = f(state + dt*k3, *args, **kwargs)
    return state + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def integrate_lorenz(T_steps=240_000, dt=5e-4, rho=170.0, k=0.0, x0=(1.2,-1.3,18.0)):
    """Integrate Lorenz with URT-like bounded feedback. Returns (trajectory, dt)."""
    x = np.empty((T_steps, 3), dtype=float)
    s = np.array(x0, dtype=float)
    for i in range(T_steps):
        s = rk4_step(lorenz_rhs, s, dt, 10.0, rho, 8.0/3.0, k, 25.0)
        x[i] = s
    cut = int(0.2*T_steps)   # drop warmup
    return x[cut:], dt

# =========================
# 2) Benettin largest Lyapunov exponent (stable, robust)
# =========================
def largest_lyapunov_benettin(traj, dt, rho=170.0, k=0.0, renorm_every=10):
    """
    Evolve a tangent vector via Jacobian along the *given trajectory* and accumulate growth.
    """
    v = np.array([1.0, 0.0, 0.0], dtype=float)
    v /= np.linalg.norm(v)
    logs = []
    for i in range(1, len(traj)):
        s = traj[i]
        J = lorenz_jacobian(s, 10.0, rho, 8.0/3.0, k, 25.0)
        # simple tangent Euler (small dt is fine here)
        v = v + dt * (J @ v)
        if (i % renorm_every) == 0:
            ln = np.linalg.norm(v)
            if ln <= 1e-20:
                v = np.array([1.0, 0.0, 0.0], dtype=float)
                continue
            logs.append(np.log(ln))
            v /= ln
    if len(logs) < 8:
        return np.nan
    return (np.sum(logs)/len(logs)) / (renorm_every*dt)

def DKY_from_l1(l1):
    return 1.0 + max(l1, 0.0)

# =========================
# 3) τ from laminar dwell-times near the two Lorenz wings
# =========================
def tau_laminar_adaptive(X, rho, p_in=0.35, hyster=1.45, min_episodes=50, tail_keep=200):
    beta = 8.0/3.0
    x0 = sqrt(beta*(rho-1.0))
    Cplus  = np.array([ x0,  x0,  rho-1.0])
    Cminus = np.array([-x0, -x0,  rho-1.0])

    dplus  = np.linalg.norm(X - Cplus , axis=1)
    dminus = np.linalg.norm(X - Cminus, axis=1)
    d = np.minimum(dplus, dminus)

    eps_in  = np.quantile(d, p_in)
    eps_out = eps_in * hyster

    # hysteretic laminar mask
    mask = np.zeros_like(d, dtype=bool)
    on = False
    for i in range(len(d)):
        if not on and d[i] < eps_in:
            on = True
        elif on and d[i] > eps_out:
            on = False
        mask[i] = on

    runs = []
    c = 0
    for v in mask:
        if v: c += 1
        else:
            if c>0:
                runs.append(c)
                c = 0
    if c>0:
        runs.append(c)

    if len(runs) < min_episodes:
        return np.nan

    runs = np.array(sorted(runs))
    tail = runs[-min(tail_keep, len(runs)):]
    if len(tail) < 10:
        return np.nan

    # log-CCDF slope on upper tail
    x = np.log(tail + 1e-9)
    ranks = np.arange(len(tail), 0, -1).astype(float)
    y = np.log(ranks / ranks[0] + 1e-12)
    m, b = np.polyfit(x, y, 1)
    tau = 1 - m     # CCDF ~ x^{-(tau-1)} ⇒ slope = -(tau-1)
    return float(tau)

def delta_metric(DKY, tau):
    if not (np.isfinite(DKY) and np.isfinite(tau)):
        return np.nan
    return (DKY - 1.0) * (tau - 2.0)

# =========================
# 4) Fine scan across ρ and k, print top bounded-chaos candidates
# =========================
Result = namedtuple("Result", "rho k l1 DKY tau delta n_ok")

rhos = np.round(np.arange(165.0, 175.0+1e-9, 0.2), 2)
ks   = np.linspace(0.0, 0.040, 21)

results = []
print("\nFine scan: rho ∈ [165.0, 175.0], step 0.2; k ∈ [0.00, 0.04], 21 points each.")
print("This uses RK4 + Benettin λ1 + adaptive laminar τ.\n")

for ri, rho in enumerate(rhos, 1):
    print(f"[{ri:02d}/{len(rhos)}] ρ = {rho:.2f}")
    for k in ks:
        # integrate trajectory
        X, dt = integrate_lorenz(T_steps=240_000, dt=5e-4, rho=rho, k=k, x0=(1.2,-1.3,18.0))
        # largest Lyapunov exponent
        l1 = largest_lyapunov_benettin(X, dt, rho=rho, k=k, renorm_every=10)
        DKY = DKY_from_l1(l1)
        tau = np.nan
        dlt = np.nan
        if np.isfinite(l1) and l1 > 0.0:
            tau = tau_laminar_adaptive(X, rho=rho, p_in=0.35, hyster=1.45, min_episodes=60, tail_keep=250)
            if np.isfinite(tau):
                dlt = delta_metric(DKY, tau)
        results.append(Result(rho, float(k), float(l1) if np.isfinite(l1) else np.nan,
                              float(DKY) if np.isfinite(DKY) else np.nan,
                              float(tau) if np.isfinite(tau) else np.nan,
                              float(dlt) if np.isfinite(dlt) else np.nan,
                              1 if (np.isfinite(l1) and l1>0 and np.isfinite(tau)) else 0))
    # progress line for this rho
    line = [r for r in results if abs(r.rho - rho) < 1e-9]
    have = sum(r.n_ok for r in line)
    print(f"  → valid chaos+τ rows at this ρ: {have}/{len(ks)}")

# =========================
# 5) Rank candidates and print the best
# =========================
def score(r):
    if r.n_ok == 0:
        return 1e9
    # distance to δ=0 and τ=2 (bounded-chaos manifold) with mild weights
    return abs(r.delta) + 0.25*abs(r.tau - 2.0)

valid = [r for r in results if r.n_ok == 1]
valid_sorted = sorted(valid, key=score)

print("\nTop 20 bounded-chaos candidates (λ1>0, τ finite), ranked by |δ| + 0.25|τ-2|:\n")
print("rank  rho      k       λ1         DKY        τ          δ")
print("----  -------  ------  ---------- ---------- ---------- ----------")
for i, r in enumerate(valid_sorted[:20], 1):
    print(f"{i:>4}  {r.rho:7.2f}  {r.k:6.3f}  {r.l1:10.5f} {r.DKY:10.5f} {r.tau:10.3f} {r.delta:10.3f}")

# =========================
# 6) Save to CSV for later analysis/plotting
# =========================
import csv, os
out_fn = "delta_law_fine_scan.csv"
with open(out_fn, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["rho","k","lambda1","DKY","tau","delta","is_valid"])
    for r in results:
        w.writerow([r.rho, r.k, r.l1, r.DKY, r.tau, r.delta, r.n_ok])
print(f"\nSaved full table → {os.path.abspath(out_fn)}")


Fine scan: rho ∈ [165.0, 175.0], step 0.2; k ∈ [0.00, 0.04], 21 points each.
This uses RK4 + Benettin λ1 + adaptive laminar τ.

[01/51] ρ = 165.00
  → valid chaos+τ rows at this ρ: 21/21
[02/51] ρ = 165.20
  → valid chaos+τ rows at this ρ: 21/21
[03/51] ρ = 165.40
  → valid chaos+τ rows at this ρ: 21/21
[04/51] ρ = 165.60
  → valid chaos+τ rows at this ρ: 21/21
[05/51] ρ = 165.80
  → valid chaos+τ rows at this ρ: 21/21
[06/51] ρ = 166.00
  → valid chaos+τ rows at this ρ: 21/21
[07/51] ρ = 166.20
  → valid chaos+τ rows at this ρ: 21/21
[08/51] ρ = 166.40
  → valid chaos+τ rows at this ρ: 21/21
[09/51] ρ = 166.60
  → valid chaos+τ rows at this ρ: 21/21
[10/51] ρ = 166.80
  → valid chaos+τ rows at this ρ: 21/21
[11/51] ρ = 167.00
  → valid chaos+τ rows at this ρ: 21/21
[12/51] ρ = 167.20
  → valid chaos+τ rows at this ρ: 21/21
[13/51] ρ = 167.40
  → valid chaos+τ rows at this ρ: 21/21
[14/51] ρ = 167.60
  → valid chaos+τ rows at this ρ: 21/21
[15/51] ρ = 167.80
  → valid chaos+τ rows at 

In [ ]:
# === δ-ridge refinement + τ bootstrap + heatmap (self-adapting) ===
import os, time, csv, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_IN  = "delta_law_fine_scan.csv"     # your full fine-scan output
CSV_OUT = "delta_law_refined_ridge.csv" # refined ridge results
HEATMAP_PNG = "delta_law_heatmap_ridge.png"

# -------------------------
# 0) Resolve measurement call (A or B or custom)
# -------------------------
def _resolve_measure():
    g = globals()
    # Option A: integrate_and_measure(rho,k) -> dict with keys lambda1, DKY, tau, delta
    if "integrate_and_measure" in g and callable(g["integrate_and_measure"]):
        def call(rho, k, T=None, dt=None, evt_q=None):
            d = g["integrate_and_measure"](rho, k)
            return float(d["lambda1"]), float(d["DKY"]), float(d["tau"]), float(d["delta"])
        return call, "A:integrate_and_measure"
    # Option B: measure_once(rho,k, T=..., dt=..., evt_q=...) -> (lambda1, DKY, tau, delta)
    if "measure_once" in g and callable(g["measure_once"]):
        def call(rho, k, T=None, dt=None, evt_q=None):
            kwargs = {}
            if T is not None: kwargs["T"] = T
            if dt is not None: kwargs["dt"] = dt
            if evt_q is not None: kwargs["evt_q"] = evt_q
            l1, DKY, tau, delt = g["measure_once"](rho, k, **kwargs)
            return float(l1), float(DKY), float(tau), float(delt)
        return call, "B:measure_once"
    raise RuntimeError(
        "Could not find a measurement function.\n"
        "Define either integrate_and_measure(rho,k)->dict or measure_once(rho,k,T,dt,evt_q)->tuple."
    )

measure_call, backend = _resolve_measure()
print(f"[resolver] using backend: {backend}")

# -------------------------
# 1) Load coarse surface and pick candidate per ρ
# -------------------------
if not os.path.exists(CSV_IN):
    raise FileNotFoundError(f"Missing {CSV_IN}. Run your fine scan first.")

df = pd.read_csv(CSV_IN)
# Filter to finite rows with λ1>0 (chaos) and τ finite
df_f = df.replace([np.inf,-np.inf], np.nan).dropna(subset=["l1","DKY","tau","delta"])
df_f = df_f[df_f["l1"] > 0]

# Score function: prioritize δ≈0 and τ≈2
def score_row(row, delta_weight=1.0, tau_weight=0.25):
    dlt = row["delta"]
    tau = row["tau"]
    return abs(dlt)*delta_weight + abs(tau - 2.0)*tau_weight

df_f["score"] = df_f.apply(score_row, axis=1)
# Pick best k per rho as seed
seeds = df_f.loc[df_f.groupby("rho")["score"].idxmin()].copy()
seeds = seeds.sort_values("rho").reset_index(drop=True)

print(f"[seeds] candidate rows selected for {seeds['rho'].nunique()} rho-slices.")

# -------------------------
# 2) Refine k via bisection to |δ|<=tol or max_iters
# -------------------------
DELTA_TOL  = 0.02     # tighter than the scan
MAX_ITERS  = 14       # ~2^-14 ≈ 6e-5 resolution if function is well-behaved
DT_REFINE  = 1.0e-3   # reuse your coarse dt; change if you prefer
T_REFINE   = None     # use your measure_once default unless you want to force longer
EVT_Q      = 0.93     # same EVT quantile you used in the fine scan

def refine_one_rho(rho, k0):
    # bracket in [k0-0.01, k0+0.01]
    kL = max(0.0, k0 - 0.01)
    kR = min(0.04, k0 + 0.01)

    # Evaluate ends
    _,_,_,dL = measure_call(rho, kL, T=T_REFINE, dt=DT_REFINE, evt_q=EVT_Q)
    _,_,_,dR = measure_call(rho, kR, T=T_REFINE, dt=DT_REFINE, evt_q=EVT_Q)

    # If signs are the same, keep the best of (kL,k0,kR); otherwise bisection
    try_bisect = (np.isfinite(dL) and np.isfinite(dR) and np.sign(dL)!=np.sign(dR))
    best = None

    def upd_best(k):
        nonlocal best
        l1,DKY,tau,dlt = measure_call(rho, k, T=T_REFINE, dt=DT_REFINE, evt_q=EVT_Q)
        if not (np.isfinite(l1) and np.isfinite(DKY) and np.isfinite(tau) and np.isfinite(dlt)):
            return
        s = abs(dlt) + 0.25*abs(tau-2.0)
        if (best is None) or (s < best["score"]):
            best = dict(rho=rho, k=k, l1=l1, DKY=DKY, tau=tau, delta=dlt, score=s)
    # Always sample the seed k0
    upd_best(k0)
    # If we can bisect, do it; else sample a small 5-point neighborhood
    if try_bisect:
        for _ in range(MAX_ITERS):
            km = 0.5*(kL+kR)
            l1,DKY,tau,dlt = measure_call(rho, km, T=T_REFINE, dt=DT_REFINE, evt_q=EVT_Q)
            s = abs(dlt) + 0.25*abs(tau-2.0)
            if (best is None) or (s < best["score"]):
                best = dict(rho=rho, k=km, l1=l1, DKY=DKY, tau=tau, delta=dlt, score=s)
            if not np.isfinite(dlt):  # bail if ill-posed
                break
            if abs(dlt) <= DELTA_TOL and l1 > 0:
                break
            # standard bisection on sign(δ)
            if np.sign(dlt) == np.sign(dL):
                kL, dL = km, dlt
            else:
                kR, dR = km, dlt
    else:
        ks = np.linspace(max(0.0,k0-0.008), min(0.04,k0+0.008), 5)
        for k in ks: upd_best(k)

    return best

t0 = time.time()
refined = []
for _, r in seeds.iterrows():
    rec = refine_one_rho(float(r["rho"]), float(r["k"]))
    if rec is not None:
        refined.append(rec)
dt = time.time()-t0
print(f"[refine] refined {len(refined)} ridge points in {dt:.1f}s")

ridge_df = pd.DataFrame(refined)
ridge_df = ridge_df.sort_values("rho")
ridge_df.to_csv(CSV_OUT, index=False)
print(f"[save] refined ridge → {CSV_OUT}")

# -------------------------
# 3) Bootstrap τ at ridge points (parametric: repeat measurement)
# -------------------------
N_BOOT = 200   # bump to 1000 if you want; this is the time-cost knob
def bootstrap_tau(rho, k, n=N_BOOT):
    taus = []
    for _ in range(n):
        l1,DKY,tau,dlt = measure_call(rho, k, T=T_REFINE, dt=DT_REFINE, evt_q=EVT_Q)
        if np.isfinite(tau): taus.append(tau)
    if len(taus) < max(10, int(0.2*n)):
        return np.nan, np.nan, np.nan
    taus = np.array(taus)
    lo, hi = np.percentile(taus, [2.5, 97.5])
    return float(np.mean(taus)), float(lo), float(hi)

print("[boot] bootstrapping τ CIs on refined ridge…")
taus_mean, taus_lo, taus_hi = [], [], []
for _, r in ridge_df.iterrows():
    tm, lo, hi = bootstrap_tau(float(r["rho"]), float(r["k"]), N_BOOT)
    taus_mean.append(tm); taus_lo.append(lo); taus_hi.append(hi)

ridge_df["tau_mean"] = taus_mean
ridge_df["tau_lo"]   = taus_lo
ridge_df["tau_hi"]   = taus_hi
ridge_df.to_csv(CSV_OUT, index=False)
print(f"[save] ridge + τ CIs → {CSV_OUT}")

# -------------------------
# 4) Heatmap of δ(ρ,k) with ridge overlay
# -------------------------
# Build a dense grid table from your coarse df_f (full scan)
tab = df_f.pivot_table(index="rho", columns="k", values="delta", aggfunc="mean")
tab = tab.sort_index().sort_index(axis=1)

plt.figure(figsize=(8,6))
im = plt.imshow(tab.values, aspect="auto", origin="lower",
                extent=[tab.columns.min(), tab.columns.max(),
                        tab.index.min(), tab.index.max()])
plt.colorbar(im, label="δ")
plt.contour(tab.columns.values, tab.index.values, tab.values,
            levels=[0.0], linewidths=1.5)

# overlay refined ridge
plt.plot(ridge_df["k"].values, ridge_df["rho"].values, 'k.', ms=4, label="refined ridge")

plt.xlabel("k (URTF gain)")
plt.ylabel("ρ (Lorenz parameter)")
plt.title("δ(ρ,k) heatmap with δ=0 contour and refined ridge")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(HEATMAP_PNG, dpi=140)
print(f"[plot] saved {HEATMAP_PNG}")

# quick text summary
close = ridge_df[np.abs(ridge_df["delta"]) <= 0.02]
print("\n=== Refined bounded-chaos hits (|δ| ≤ 0.02 & λ1>0) ===")
for _, r in close.iterrows():
    print(f"ρ={r['rho']:.2f}  k={r['k']:.4f}  λ1={r['l1']:.4f}  τ={r['tau']:.3f}  "
          f"δ={r['delta']:.4f}  τCI=[{r['tau_lo']:.3f},{r['tau_hi']:.3f}]")

RuntimeError: Could not find a measurement function.
Define either integrate_and_measure(rho,k)->dict or measure_once(rho,k,T,dt,evt_q)->tuple.

In [ ]:
import numpy as np

# ========= ROSSER SYSTEM + BENETTIN λ1 =========
def lorenz_step(x, y, z, sigma, beta, rho, k, dt):
    # Controlled Lorenz: dx/dt += -k * (state norm)
    r = np.sqrt(x*x + y*y + z*z)
    dx = sigma * (y - x) - k * r
    dy = x * (rho - z) - y - k * r
    dz = x*y - beta*z - k * r
    return x + dt*dx, y + dt*dy, z + dt*dz

def lyapunov_benettin(lamv, x, y, z, sigma, beta, rho, k, dt):
    eps = 1e-8
    x2, y2, z2 = x + eps*lamv[0], y + eps*lamv[1], z + eps*lamv[2]
    x, y, z = lorenz_step(x, y, z, sigma, beta, rho, k, dt)
    x2, y2, z2 = lorenz_step(x2, y2, z2, sigma, beta, rho, k, dt)
    dv = np.array([x2-x, y2-y, z2-z])
    norm = np.linalg.norm(dv)
    lamv[:] = dv / norm
    return x, y, z, lamv, np.log(norm / eps)

def measure_once(rho, k, T=200000, dt=1e-3, evt_q=0.93):
    sigma, beta = 10.0, 8/3
    x=y=z=1.0
    lamv = np.array([1.0,0.0,0.0])

    # integrate
    lsum = 0.0
    for i in range(T):
        x, y, z, lamv, li = lyapunov_benettin(lamv, x, y, z, sigma, beta, rho, k, dt)
        lsum += li

    lambda1 = lsum / (T*dt)
    DKY = 1 + max(lambda1,0)

    # return-time tau estimate
    r = np.sqrt(x*x + y*y + z*z)
    thr = np.quantile(r, evt_q)
    above = r > thr
    runs = []
    cnt = 0
    for val in above:
        if val: cnt += 1
        elif cnt>0:
            runs.append(cnt)
            cnt = 0
    tau = np.mean(runs) if len(runs)>3 else np.nan

    delta = (DKY-1)*(tau-2) if tau==tau else np.nan
    return float(lambda1), float(DKY), float(tau), float(delta)

In [ ]:
measure_once(...)

TypeError: measure_once() missing 1 required positional argument: 'k'

In [ ]:
measure_once(166.0, 0.01)

TypeError: 'numpy.bool' object is not iterable

In [ ]:
def measure_once(rho, k, T=40000, dt=1e-3, evt_q=0.90):
    # Integrate Lorenz with feedback
    sol = integrate(rho=rho, k=k, T=T, dt=dt)  # returns arrays (x,y,z)
    x, y, z = sol.T
    r = np.sqrt(x*x + y*y + z*z)

    # Compute Lyapunov exponent (Benettin, already inside integrate)
    l1 = lyap_last  # integrate() must set this globally or return it
    DKY = 1 + max(l1, 0.0)

    # EVT tail-based laminar return time estimator
    dx = np.abs(np.diff(r))
    thr = np.quantile(dx, evt_q)
    mask = dx > thr

    # Count run lengths of mask==True
    runs = []
    cnt = 0
    for m in mask:
        if m:
            cnt += 1
        else:
            if cnt > 0:
                runs.append(cnt)
            cnt = 0
    if cnt > 0:
        runs.append(cnt)

    if len(runs) < 5:
        tau = np.nan
    else:
        # Hill tail estimator
        rs = np.sort(runs)
        tail = rs[-max(10, len(rs)//10):]  # top 10% or at least 10 samples
        tau = len(tail) / np.sum(np.log(tail / tail[0] + 1e-12)) + 2

    delta = (DKY - 1.0) * (tau - 2.0) if np.isfinite(tau) else np.nan

    return float(l1), float(DKY), float(tau), float(delta)

In [ ]:
measure_once(166.0, 0.01)

NameError: name 'integrate' is not defined

In [11]:
def tau_from_peaks(x, dt, q=0.85, min_prom=0.05, min_dist_time=1.0):
    from scipy.signal import find_peaks
    # enforce a minimum inter-peak distance in samples
    distance = max(1, int(min_dist_time/dt))
    pk, props = find_peaks(x, prominence=min_prom, distance=distance)
    if len(pk) < 3:
        return np.nan
    heights = x[pk]
    thr = np.quantile(heights, q)
    bursts = pk[heights >= thr]
    if len(bursts) < 3:
        return np.nan
    inter = np.diff(bursts) * dt
    return float(np.mean(inter))

In [12]:
def integrate_with_lyap(c, k, T=20000, dt=0.01, renorm_steps=50, burnin_steps=2000):
    steps = int(T/dt)
    x,y,z = 0.1, 0.1, 0.1

    # tangent vector
    dx,dy,dz = 1e-8, 0.0, 0.0
    sum_log = 0.0
    count = 0

    traj_x = []

    for i in range(steps):
        # main system (RK4 for stability)
        def f(state):
            X,Y,Z = state
            a = 0.2; b = 0.2
            return np.array([
                -Y - Z + k*Z,              # feedback on x
                X + a*Y,
                b + Z*(X - c),
            ])

        s  = np.array([x,y,z])
        s1 = s + dt * f(s)
        s2 = s + dt * f(s + 0.5*(s1 - s))
        s3 = s + dt * f(s + 0.5*(s2 - s))
        s4 = s + dt * f(s3)
        s  = s + (s1 + 2*s2 + 2*s3 + s4 - 6*s)/6.0
        x,y,z = s

        # tangent evolution (same Jacobian-free trick via two copies)
        eps = np.array([dx,dy,dz])
        s_eps = s + eps
        s1 = s_eps + dt * f(s_eps)
        s2 = s_eps + dt * f(s_eps + 0.5*(s1 - s_eps))
        s3 = s_eps + dt * f(s_eps + 0.5*(s2 - s_eps))
        s4 = s_eps + dt * f(s3)
        s_eps = s_eps + (s1 + 2*s2 + 2*s3 + s4 - 6*s_eps)/6.0
        dx,dy,dz = (s_eps - s)

        # Benettin renormalization AFTER burn-in
        if i >= burnin_steps and (i - burnin_steps + 1) % renorm_steps == 0:
            norm = np.sqrt(dx*dx + dy*dy + dz*dz)
            if norm <= 0 or not np.isfinite(norm):
                # reset tiny/NaN tangent to avoid crashes
                dx,dy,dz = 1e-8, 0.0, 0.0
                continue
            sum_log += np.log(norm)
            # reset tangent direction
            dx,dy,dz = dx/norm, dy/norm, dz/norm
            count += 1

        # keep x trace for τ only after burn-in
        if i >= burnin_steps:
            traj_x.append(x)

    # ✅ correct time normalization: per unit time
    l1 = np.nan
    if count > 0:
        l1 = sum_log / (count * renorm_steps * dt)

    return np.array(traj_x), float(l1)

In [13]:
def tau_from_peaks(x, dt, q=0.85, min_prom=0.05, min_dist_time=1.0):
    from scipy.signal import find_peaks
    # enforce a minimum inter-peak distance in samples
    distance = max(1, int(min_dist_time/dt))
    pk, props = find_peaks(x, prominence=min_prom, distance=distance)
    if len(pk) < 3:
        return np.nan
    heights = x[pk]
    thr = np.quantile(heights, q)
    bursts = pk[heights >= thr]
    if len(bursts) < 3:
        return np.nan
    inter = np.diff(bursts) * dt
    return float(np.mean(inter))

In [ ]:
import numpy as np

# ----------------------------
# Rössler ODE + RK4 Integrator
# ----------------------------
def rossler_step(x, y, z, a, b, c):
    dx = -y - z
    dy = x + a*y
    dz = b + z*(x - c)
    return dx, dy, dz

def integrate_rossler(a, b, c, T=40000, dt=1e-3, transient=2000):
    steps = int(T/dt)
    x = np.zeros(steps); y = np.zeros(steps); z = np.zeros(steps)

    # initial condition
    x[0], y[0], z[0] = 1.0, 1.0, 0.0

    for i in range(steps-1):
        dx1, dy1, dz1 = rossler_step(x[i], y[i], z[i], a, b, c)
        dx2, dy2, dz2 = rossler_step(x[i]+0.5*dt*dx1, y[i]+0.5*dt*dy1, z[i]+0.5*dt*dz1, a, b, c)
        dx3, dy3, dz3 = rossler_step(x[i]+0.5*dt*dx2, y[i]+0.5*dt*dy2, z[i]+0.5*dt*dz2, a, b, c)
        dx4, dy4, dz4 = rossler_step(x[i]+dt*dx3, y[i]+dt*dy3, z[i]+dt*dz3, a, b, c)

        x[i+1] = x[i] + (dt/6)*(dx1 + 2*dx2 + 2*dx3 + dx4)
        y[i+1] = y[i] + (dt/6)*(dy1 + 2*dy2 + 2*dy3 + dy4)
        z[i+1] = z[i] + (dt/6)*(dz1 + 2*dz2 + 2*dz3 + dz4)

    return x[transient:], y[transient:], z[transient:]

# ----------------------------------------------------
# Benettin Lyapunov Exponent (λ₁)
# ----------------------------------------------------
def lyapunov(x, y, z, dt):
    eps = 1e-8
    lsum = 0.0
    vx, vy, vz = eps, 0, 0
    for i in range(len(x)-1):
        # tangent
        dx = -y[i] - z[i]
        dy = x[i] + 0.2*y[i]
        dz = 0.2 + z[i]*(x[i] - C_CURRENT)
        norm = np.sqrt(vx*vx + vy*vy + vz*vz) + 1e-12
        vx /= norm; vy /= norm; vz /= norm
        lsum += np.log(norm)
    return lsum / ((len(x)-1) * dt)

# ----------------------------------------------------
# Laminar return time τ (adaptive threshold method)
# ----------------------------------------------------
def laminar_tau(z):
    threshold = np.percentile(z, 90)
    above = z < threshold  # Rössler laminar region is near small z
    runs = []
    count = 0
    for val in above:
        if val: count += 1
        else:
            if count > 0: runs.append(count)
            count = 0
    if len(runs) == 0: return None
    return np.mean(runs)

# ----------------------------------------------------
# δ = (D_KY − 1)(τ − 2)
# ----------------------------------------------------
def delta(DKY, tau):
    if tau is None:
        return None
    return (DKY - 1) * (tau - 2)

# ----------------------------------------------------
# MAIN SCAN (Correct crisis window!)
# ----------------------------------------------------
a = 0.2
b = 0.2

c_values = np.arange(8.3, 10.3, 0.05)     # ✅ Crisis & intermittency region
k_values = np.arange(0.00, 0.041, 0.005)  # ✅ Small feedback scan

print("Scanning Rössler bounded-chaos region...\n")

results = []

for c in c_values:
    for k in k_values:
        global C_CURRENT
        C_CURRENT = c + k  # feedback modifies effective c-parameter

        x, y, z = integrate_rossler(a, b, C_CURRENT)
        l1 = lyapunov(x, y, z, 1e-3)
        DKY = 1 + max(l1, 0)
        tau = laminar_tau(z)
        d = delta(DKY, tau)

        print(f"c={c:.3f}  k={k:.3f} | λ1={l1:.4f} DKY={DKY:.3f} τ={tau} δ={d}")
        results.append((c, k, l1, DKY, tau, d))

print("\nDone.\n")